In [ ]:
import sqlite3
from bs4 import BeautifulSoup
import json
import requests
import re
import pandas as pd
from time import *
from random import randint
import time
import configparser

: 

In [84]:
user_rating = 5
num_votes = 35000
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/106.0.0.0 Safari/537.36'}
imdbId = []

def get_imdbId(user_rating, num_votes):
    url = f"https://www.imdb.com/search/title/?title_type=feature&user_rating={user_rating},&sort=num_votes,desc&num_votes={num_votes},&count=250"
    while True:
        url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.text, "html.parser")
       
        script_tags = soup.find_all('script', {'type': 'application/json'})
        # Iterate through script tags to find the one with the JSON data
        for script_tag in script_tags:
            script_text = script_tag.string
            if script_text:
                # Parse JSON data
                data = json.loads(script_text)
                # Extract titleId from each title
                imdbId.extend(item.get('titleId', '') for item in data.get('props', {}).get('pageProps', {}).get('searchResults', {}).get('titleResults', {}).get('titleListItems', []))

        if (soup.find('a', {'class': 'lister-page-next next-page'})):
            url = 'https://www.imdb.com/' + soup.find('a', {'class': 'lister-page-next next-page'})['href']
        else:
            break


In [80]:
get_imdbId(user_rating, num_votes)

In [81]:
len(imdbId)

250

In [82]:
len(list(set(imdbId)))

250

In [135]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service as ChromeService
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import json
import time

# Set the path to the webdriver executable (e.g., chromedriver, geckodriver)
driver_path = 'C:/Users/Maja/Downloads/chromedriver-win64/chromedriver.exe'

# Set up ChromeOptions
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3")
#chrome_options.add_argument('--headless')  # Run in headless mode if needed

# Create a new instance of the Chrome WebDriver
chrome_service = Service(driver_path)
driver = webdriver.Chrome(service=chrome_service, options=chrome_options) 

# URL of the website
url = 'https://www.imdb.com/search/title/?title_type=feature&user_rating=5,10&sort=num_votes,desc&num_votes=35000,&count=250'

# Navigate to the website
driver.get(url)

# Function to scroll to the bottom of the page
def scroll_to_bottom():
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)  # Add a small delay to let the content load

# Scroll to the bottom a few times to load more content
# for _ in range(10):
#     scroll_to_bottom()

# while True:
#     try:
#         show_more_button = WebDriverWait(driver, 10).until(
#             EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'ipc-see-more__button')]"))
#         )
#         driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
#         show_more_button.click()
#         WebDriverWait(driver, 10).until(
#             EC.presence_of_element_located((By.XPATH, "//div[contains(@class, 'load-more-data')]"))
#         )
#         scroll_to_bottom()  # Scroll to the bottom again to load more content
#     except:
#         break

# Keep clicking the "Show more" button until it's no longer available
while True:
    show_more_button = WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, "//button[contains(@class, 'ipc-see-more__button')]"))
    )

    # Use JavaScript to scroll the button into view
    driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)

    try:
        # Wait for the button to be clickable and then click it
        WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, "//button[contains(@class, 'ipc-see-more__button')]"))
        ).click()
        scroll_to_bottom()  # Scroll to the bottom again to load more content
    except:
        # If the button is not clickable, break the loop
        break

# Extract the HTML content using BeautifulSoup
html_content = driver.page_source
#print(html_content)
soup = BeautifulSoup(html_content, 'html.parser')
#print(soup)

script_tags = soup.find_all('script', {'type': 'application/json'})

imdbId = []

# Iterate through script tags to find the one with the JSON data
for script_tag in script_tags:
    script_text = script_tag.string
    if script_text:
        # Parse JSON data
        data = json.loads(script_text)
        # Extract titleId from each title
        imdbId.extend(item.get('titleId', '') for item in data.get('props', {}).get('pageProps', {}).get('searchResults', {}).get('titleResults', {}).get('titleListItems', []))
    
# # Extract the JSON script using BeautifulSoup
# script_tag = soup.find_all('script', {'id': '__NEXT_DATA__'}).contents[0]
# print(script_tag)

# title_ids =[]

# if json_script is not None:
#     # Now you can parse the JSON script as needed
#     parsed_json = json.loads(script_tag)

#     # Extract titleId elements from the parsed JSON
#     title_ids = [item['titleId'] for item in parsed_json['props']['pageProps']['searchResults']['titleResults']['titleListItems']]

#     print("titleIds:", title_ids)

# else:
#     print("Script content is None. Check if the script tag contains the expected data.")

# Close the webdriver when done
driver.quit()

# # Extract and print the updated JSON script content
# updated_json_script = driver.execute_script('return document.querySelector("script#__NEXT_DATA__").innerHTML;')
# print(updated_json_script)

In [136]:
len(imdbId)

250

In [129]:
len(title_ids)

250

In [89]:
print(selenium.__version__)

4.10.0


In [88]:
import selenium

In [137]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from bs4 import BeautifulSoup
import time

# Set the path to the webdriver executable (e.g., chromedriver, geckodriver)
driver_path = 'C:/Users/Maja/Downloads/chromedriver-win64/chromedriver.exe'

# Set up ChromeOptions
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3")
#chrome_options.add_argument('--headless')  # Run in headless mode if needed

# Create a new instance of the Chrome WebDriver
chrome_service = Service(driver_path)
driver = webdriver.Chrome(service=chrome_service, options=chrome_options) 

# Navigate to the IMDb page
driver.get('https://www.imdb.com/search/title/?title_type=feature&user_rating=5,10&sort=num_votes,desc&num_votes=35000,&count=250')

# Function to extract titleId from the current page
def extract_title_ids(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    script_tags = soup.find_all('script', {'type': 'application/json'})

    for script in script_tags:
        script_content = script.string
        if script_content and 'titleId' in script_content:
            json_data = json.loads(script_content)
            title_results = json_data.get('props', {}).get('pageProps', {}).get('searchResults', {}).get('titleResults', {})
            for title_item in title_results.get('titleListItems', []):
                title_id = title_item.get('titleId')
                if title_id:
                    yield title_id

# Function to click the "Show more" button
def click_show_more():
    show_more_button = driver.find_element(By.XPATH, '//button[contains(@class, "ipc-see-more__button")]')
    show_more_button.click()

# Extract titles until there are no more "Show more" buttons
while True:
    # Wait for the content to load
    time.sleep(2)

    # Extract the HTML content using BeautifulSoup
    html_content = driver.page_source

    # Extract titleIds from the current page
    title_ids = list(extract_title_ids(html_content))

    # Output or process titleIds as needed
    print(title_ids)

    # Check if there is a "Show more" button
    try:
        click_show_more()
    except:
        # No more "Show more" button, break the loop
        break

# Close the WebDriver
driver.quit()

['tt0111161', 'tt0468569', 'tt1375666', 'tt0137523', 'tt0109830', 'tt0110912', 'tt0816692', 'tt0133093', 'tt0068646', 'tt0120737', 'tt0167260', 'tt1345836', 'tt0114369', 'tt0167261', 'tt1853728', 'tt0172495', 'tt0372784', 'tt0361748', 'tt0993846', 'tt0102926', 'tt0120815', 'tt0848228', 'tt7286456', 'tt0076759', 'tt0108052', 'tt1130884', 'tt0482571', 'tt0407887', 'tt0120689', 'tt0499549', 'tt0080684', 'tt0071562', 'tt0209144', 'tt0088763', 'tt0120338', 'tt2015381', 'tt4154796', 'tt0099685', 'tt0110413', 'tt0169547', 'tt0325980', 'tt0910970', 'tt4154756', 'tt0266697', 'tt0120586', 'tt0120382', 'tt0434409', 'tt0103064', 'tt0114814', 'tt0110357', 'tt0371746', 'tt1049413', 'tt0086190', 'tt1431045', 'tt0266543', 'tt0081505', 'tt0112573', 'tt0105236', 'tt0264464', 'tt1392190', 'tt0338013', 'tt0073486', 'tt0114709', 'tt0107290', 'tt2267998', 'tt0119217', 'tt0477348', 'tt0167404', 'tt0082971', 'tt1392170', 'tt0268978', 'tt2488496', 'tt0198781', 'tt2582802', 'tt0078748', 'tt0095016', 'tt1201607'

In [138]:
len(title_ids)

250

In [141]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time

# Set the path to your ChromeDriver executable
driver_path = 'C:/Users/Maja/Downloads/chromedriver-win64/chromedriver.exe'

# Set up ChromeOptions
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3")
## Run in headless mode if needed
#chrome_options.add_argument('--headless')  

# Create a new instance of the Chrome WebDriver
chrome_service = Service(driver_path)
driver = webdriver.Chrome(service=chrome_service, options=chrome_options) 

# Navigate to the IMDb page
driver.get('https://www.imdb.com/search/title/?title_type=feature&user_rating=5,10&sort=num_votes,desc&num_votes=35000,&count=250')

# Function to extract tt from the link
def extract_tt(link):
    href = link.get('href')
    if href and '/title/' in href:
        return href.split('/title/')[-1].split('/')[0]

# Function to extract tt from the current page
def extract_tts(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    link_tags = soup.find_all('a', {'class': 'ipc-title-link-wrapper'})
    return [extract_tt(link) for link in link_tags]

# Function to click the "See more" button using JavaScript
def click_see_more():
    driver.execute_script("document.querySelector('.ipc-see-more__button').click();")

# Extract tt until there are no more "See more" buttons
tt_list = []
while True:
    # Wait for the content to load
    time.sleep(2)

    # Extract tt from the current page
    tt_list.extend(extract_tts(driver.page_source))

    # Check if there is a "See more" button
    try:
        click_see_more()
    except:
        # No more "See more" button, break the loop
        break

# Close the WebDriver
driver.quit()

# Display the extracted tt list
print(tt_list)

['tt0111161', 'tt0468569', 'tt1375666', 'tt0137523', 'tt0109830', 'tt0110912', 'tt0816692', 'tt0133093', 'tt0068646', 'tt0120737', 'tt0167260', 'tt1345836', 'tt0114369', 'tt0167261', 'tt1853728', 'tt0172495', 'tt0372784', 'tt0361748', 'tt0993846', 'tt0102926', 'tt0120815', 'tt0848228', 'tt7286456', 'tt0076759', 'tt0108052', 'tt1130884', 'tt0482571', 'tt0407887', 'tt0120689', 'tt0499549', 'tt0080684', 'tt0071562', 'tt0209144', 'tt0088763', 'tt0120338', 'tt2015381', 'tt4154796', 'tt0099685', 'tt0110413', 'tt0169547', 'tt0325980', 'tt0910970', 'tt4154756', 'tt0266697', 'tt0120586', 'tt0120382', 'tt0434409', 'tt0103064', 'tt0114814', 'tt0110357', 'tt0371746', 'tt1049413', 'tt0086190', 'tt1431045', 'tt0266543', 'tt0081505', 'tt0112573', 'tt0105236', 'tt0264464', 'tt1392190', 'tt0338013', 'tt0073486', 'tt0114709', 'tt0107290', 'tt2267998', 'tt0119217', 'tt0477348', 'tt0167404', 'tt0082971', 'tt1392170', 'tt0268978', 'tt2488496', 'tt0198781', 'tt2582802', 'tt0078748', 'tt0095016', 'tt1201607'

In [142]:
len(tt_list)

53250

In [143]:
len(set(tt_list))

5000

In [145]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import time

# Set the path to your ChromeDriver executable
driver_path = 'C:/Users/Maja/Downloads/chromedriver-win64/chromedriver.exe'

# Set up ChromeOptions
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3")
## Run in headless mode if needed
#chrome_options.add_argument('--headless')  

# Create a new instance of the Chrome WebDriver
chrome_service = Service(driver_path)
driver = webdriver.Chrome(service=chrome_service, options=chrome_options) 

# Navigate to the IMDb page
driver.get('https://www.imdb.com/search/title/?title_type=feature&user_rating=5,10&sort=num_votes,desc&num_votes=35000,&count=250')

# Function to extract tt from the link
def extract_tt(link):
    href = link.get('href')
    if href and '/title/' in href:
        return href.split('/title/')[-1].split('/')[0]

# Function to extract tt from the current page
def extract_tts(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    link_tags = soup.select('.ipc-title-link-wrapper[href^="/title/"]')
    return [extract_tt(link) for link in link_tags]

# Function to click the "See more" button using JavaScript
def click_see_more():
    driver.execute_script("document.querySelector('.ipc-see-more__button').click();")

# Extract tt until there are no more "See more" buttons
tt_set = set()  # Use a set to avoid duplicates
while True:
    # Wait for the content to load
    time.sleep(2)

    # Extract tt from the current page
    tt_set.update(extract_tts(driver.page_source))

    # Check if there is a "See more" button
    try:
        click_see_more()
    except:
        # No more "See more" button, break the loop
        break

# Close the WebDriver
driver.quit()

# Convert the set to a list for display
tt_list = list(tt_set)

# Display the extracted tt list
print(tt_list)

['tt2548396', 'tt0097441', 'tt0878804', 'tt1078188', 'tt0368891', 'tt6820256', 'tt11145118', 'tt0918927', 'tt0472160', 'tt0066808', 'tt3411444', 'tt14128670', 'tt0087928', 'tt2262227', 'tt1232829', 'tt0347304', 'tt4701182', 'tt2404311', 'tt0109635', 'tt0356150', 'tt10696896', 'tt2823054', 'tt2273657', 'tt5719748', 'tt3405236', 'tt0120768', 'tt0058182', 'tt0492956', 'tt0364970', 'tt5638642', 'tt0110622', 'tt0327162', 'tt0971209', 'tt0120873', 'tt2278871', 'tt1255953', 'tt0760329', 'tt1213641', 'tt1174732', 'tt0410377', 'tt0187078', 'tt9032400', 'tt1205489', 'tt0861689', 'tt3808342', 'tt0172495', 'tt4196776', 'tt0180093', 'tt2763304', 'tt0058461', 'tt0416320', 'tt0217869', 'tt1464580', 'tt13751694', 'tt2788710', 'tt1464540', 'tt2334649', 'tt0117998', 'tt1620981', 'tt0061395', 'tt1263670', 'tt0118694', 'tt0337692', 'tt0119190', 'tt0103956', 'tt0107076', 'tt1872194', 'tt1567437', 'tt0362270', 'tt1764183', 'tt0960731', 'tt10752004', 'tt7146812', 'tt2726560', 'tt0145660', 'tt0077651', 'tt010

In [146]:
len(tt_list)

5000

In [147]:
import api_keys

In [148]:
movies = []

def get_tmdb_movie_data(api_key, imdbId):
    for movie_id in tt_list:  
        url = f'https://api.themoviedb.org/3/movie/{movie_id}?api_key={api_key}&language=en-US'
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            movies.append(soup)
        else:
            print(f"Error: {response.status_code}")

In [149]:
get_tmdb_movie_data(api_keys.tmdb_api_key, tt_list)

In [151]:
len(movies)

5000

In [156]:
credits = []

def get_tmdb_credits_data(api_key, imdbId):
    for movie_id in tt_list:  
        url = f'https://api.themoviedb.org/3/movie/{movie_id}/credits?api_key={api_key}&language=en-US'
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            credits.append(soup)
        else:
            print(f"Error: {response.status_code}")

In [157]:
get_tmdb_credits_data(api_keys.tmdb_api_key, tt_list)

In [159]:
len(credits)

5000

In [163]:
keywords = []

def get_tmdb_keywords_data(api_key, imdbId):
    for movie_id in tt_list:  
        url = f'https://api.themoviedb.org/3/movie/{movie_id}/keywords?api_key={api_key}&language=en-US'
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            keywords.append(soup)
        else:
            print(f"Error: {response.status_code}")

In [164]:
get_tmdb_keywords_data(api_keys.tmdb_api_key, tt_list)

c:\ProgramData\Anaconda3\lib\site-packages\bs4\__init__.py:435: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  warnings.warn(


In [165]:
len(keywords)

5000

In [167]:
movies[1]

{"adult":false,"backdrop_path":"/eW8c4u3WhOVkPMnfe294lXggO3i.jpg","belongs_to_collection":null,"budget":18000000,"genres":[{"id":18,"name":"Drama"},{"id":36,"name":"History"},{"id":10752,"name":"War"}],"homepage":"","id":9665,"imdb_id":"tt0097441","original_language":"en","original_title":"Glory","overview":"Robert Gould Shaw leads the US Civil War's first all-black volunteer company, fighting prejudices of both his own Union army and the Confederates.","popularity":20.379,"poster_path":"/tubPIJJah9iJ9eDHXxaxjLdOcQd.jpg","production_companies":[{"id":27349,"logo_path":null,"name":"Freddie Fields Productions","origin_country":"US"},{"id":559,"logo_path":"/eC0bWHVjnjUducyA6YFoEFqnPMC.png","name":"TriStar Pictures","origin_country":"US"}],"production_countries":[{"iso_3166_1":"US","name":"United States of America"}],"release_date":"1989-12-15","revenue":26800000,"runtime":122,"spoken_languages":[{"english_name":"English","iso_639_1":"en","name":"English"}],"status":"Released","tagline":"T

In [178]:
type(movies[0])

bs4.BeautifulSoup

In [175]:
aaa = movies[0].get("imdb_id", 3)
print(aaa)

3


In [254]:
import json

# Define a function to extract relevant data from a BeautifulSoup object
def extract_movie_data(movie_soup):
    try:
        # Extract the JSON string
        json_string = movie_soup.text
        
        # Convert JSON string to a dictionary
        data_dict = json.loads(json_string)

        # Extract relevant data
        id = data_dict.get('id')
        imdb_id = data_dict.get('imdb_id')
        title = data_dict.get('title')
        release_date = data_dict.get('release_date')
        vote_average = data_dict.get('vote_average')
        genres = ", ".join(genre['name'] for genre in data_dict['genres'])
        overview = data_dict.get('overview')
        tagline = data_dict.get('tagline')
        popularity = data_dict.get('popularity')
        revenue = data_dict.get('revenue')
        runtime = data_dict.get('runtime')
        spoken_languages = ", ".join(language['english_name'] for language in data_dict['spoken_languages'])
        vote_count = data_dict.get('vote_count')

        return {
            'id': id,
            'imdb_id': imdb_id,
            'title': title,
            'release_date': release_date,
            'vote_average': vote_average,
            'genres': genres,
            'overview': overview,
            'tagline': tagline,
            'popularity': popularity,
            'revenue': revenue,
            'runtime': runtime,
            'spoken_languages': spoken_languages,
            'vote_count': vote_count
        }
    except Exception as e:
        print(f"Error extracting data: {e}")
        return None

# Loop to extract data for each movie
movies_metadata = []
for movie_soup in movies:
    movie_metadata = extract_movie_data(movie_soup)
    if movie_metadata:
        movies_metadata.append(movie_metadata)

# Now 'movies_data' is a list of dictionaries, each containing movie information


In [255]:
movies_metadata[0]

{'id': 384521,
 'imdb_id': 'tt2548396',
 'title': 'The Cloverfield Paradox',
 'release_date': '2018-02-04',
 'vote_average': 5.598,
 'genres': 'Horror, Science Fiction, Action, Thriller',
 'overview': 'Orbiting above a planet on the brink of war, scientists test a device to solve an energy crisis and end up face-to-face with a dark alternate reality.',
 'tagline': 'The future unleashed every thing',
 'popularity': 18.481,
 'revenue': 0,
 'runtime': 102,
 'spoken_languages': 'English',
 'vote_count': 3057}

In [1]:
movie_metadata.head()

NameError: name 'movie_metadata' is not defined

In [272]:
import pandas as pd

movie_metadata = pd.DataFrame(movies_metadata)

movie_metadata.head()

,id,imdb_id,title,release_date,vote_average,genres,overview,tagline,popularity,revenue,runtime,spoken_languages,vote_count
0,384521,tt2548396,The Cloverfield Paradox,2018-02-04,5.598,"Horror, Science Fiction, Action, Thriller","Orbiting above a planet on the brink of war, s...",The future unleashed every thing,18.481,0,102,English,3057
1,9665,tt0097441,Glory,1989-12-15,7.485,"Drama, History, War",Robert Gould Shaw leads the US Civil War's fir...,Their innocence. Their heritage. Their lives.,20.379,26800000,122,English,1475
2,22881,tt0878804,The Blind Side,2009-11-20,7.693,Drama,"The story of Michael Oher, a homeless and trau...",Based on the extraordinary true story,34.286,309200000,129,English,5972
3,14748,tt1078188,Boy A,2007-10-28,7.090,"Crime, Drama",Freed after a lengthy term in a juvenile deten...,Who decides who gets a second chance?,7.818,1200398,102,English,421
4,2059,tt0368891,National Treasure,2004-11-19,6.621,"Adventure, Action, Thriller, Mystery","Modern treasure hunters, led by archaeologist ...",The greatest adventure history has ever revealed.,28.629,347500000,131,"Latin, English, Spanish",5959


In [257]:
movie_metadata.dtypes

id                    int64
imdb_id              object
title                object
release_date         object
vote_average        float64
genres               object
overview             object
tagline              object
popularity          float64
revenue               int64
runtime               int64
spoken_languages     object
vote_count            int64
dtype: object

In [258]:
movie_metadata.isna().sum()

id                  0
imdb_id             0
title               0
release_date        0
vote_average        0
genres              0
overview            0
tagline             0
popularity          0
revenue             0
runtime             0
spoken_languages    0
vote_count          0
dtype: int64

In [259]:
movie_metadata['release_date'].head()

0    2018-02-04
1    1989-12-15
2    2009-11-20
3    2007-10-28
4    2004-11-19
Name: release_date, dtype: object

In [260]:
movie_metadata['release_date'] = pd.to_datetime(movie_metadata['release_date'])

In [261]:
movie_metadata['release_year'] = movie_metadata['release_date'].dt.year

In [262]:
movie_metadata['release_year']

0       2018
1       1989
2       2009
3       2007
4       2004
        ... 
4995    1998
4996    2002
4997    2006
4998    2015
4999    1995
Name: release_year, Length: 5000, dtype: int64

In [248]:
# from sqlalchemy import create_engine

# # Path to the SQLite database
# engine = create_engine('sqlite:///movie_database.db')

# # Convert lists to strings
# metadata['genres'] = metadata['genres'].apply(lambda x: ', '.join(x) if x else '')
# metadata['spoken_languages'] = metadata['spoken_languages'].apply(lambda x: ', '.join(x) if x else '')

# # Save the DataFrame to the 'movies' table in the SQLite database
# with engine.connect() as connection:
#     metadata.to_sql('movies', con=connection, if_exists='replace', index=False)

In [219]:
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK stopwords (you only need to do this once)
nltk.download('stopwords')
nltk.download('punkt')

# Handle missing values if any
movie_metadata['overview'].fillna('', inplace=True)
movie_metadata['tagline'].fillna('', inplace=True)

# Remove non-alphabetic characters
movie_metadata['overview'] = movie_metadata['overview'].replace(r'[^a-zA-Z]', ' ', regex=True)
movie_metadata['tagline'] = movie_metadata['tagline'].replace(r'[^a-zA-Z]', ' ', regex=True)

# Convert to lowercase
movie_metadata['overview'] = movie_metadata['overview'].str.lower()
movie_metadata['tagline'] = movie_metadata['tagline'].str.lower()

# Remove English stopwords
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word.lower() not in stop_words]
    return ' '.join(filtered_text)

# Apply stopword removal to 'overview' and 'tagline'
movie_metadata['overview'] = movie_metadata['overview'].apply(remove_stopwords)
movie_metadata['tagline'] = movie_metadata['tagline'].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Maja\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Maja\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [222]:
from nltk.stem import PorterStemmer

# Stemming using Porter Stemmer
stemmer = PorterStemmer()

def apply_stemming(text):
    stemmed_text = [stemmer.stem(word) for word in word_tokenize(text)]
    return ' '.join(stemmed_text)

# Apply stemming to 'overview' and 'tagline'
movie_metadata['overview'] = movie_metadata['overview'].apply(apply_stemming)
movie_metadata['tagline'] = movie_metadata['tagline'].apply(apply_stemming)

In [223]:
movie_metadata[['overview', 'tagline']]

,overview,tagline
0,orbit planet brink war scientist test devic so...,futur unleash everi thing
1,robert gould shaw lead us civil war first blac...,innoc heritag live
2,stori michael oher homeless traumat boy becam ...,base extraordinari true stori
3,freed lengthi term juvenil detent center convi...,decid get second chanc
4,modern treasur hunter led archaeologist ben ga...,greatest adventur histori ever reveal
...,...,...
4995,success physician devot famili man john dolitt...,talk anim
4996,egyptian queen cleopatra bet roman emperor jul...,funniest film ancient histori
4997,london high societi mous roddi flush toilet si...,someon go
4998,small town colleg campu philosophi professor e...,


In [237]:
credits[1]

{"id":9665,"cast":[{"adult":false,"gender":2,"id":4756,"known_for_department":"Acting","name":"Matthew Broderick","original_name":"Matthew Broderick","popularity":28.489,"profile_path":"/papqFgpyroZJEqd7WvuNGN8ti2k.jpg","cast_id":1,"character":"Col. Robert Gould Shaw","credit_id":"52fe4518c3a36847f80bc2d3","order":0},{"adult":false,"gender":2,"id":5292,"known_for_department":"Acting","name":"Denzel Washington","original_name":"Denzel Washington","popularity":63.542,"profile_path":"/jj2Gcobpopokal0YstuCQW0ldJ4.jpg","cast_id":2,"character":"Pvt. Trip","credit_id":"52fe4518c3a36847f80bc2d7","order":1},{"adult":false,"gender":2,"id":2130,"known_for_department":"Acting","name":"Cary Elwes","original_name":"Cary Elwes","popularity":27.902,"profile_path":"/9P0JD0LC4j3Y43s6TGM8rOqmbwb.jpg","cast_id":3,"character":"Maj. Cabot Forbes","credit_id":"52fe4518c3a36847f80bc2db","order":2},{"adult":false,"gender":2,"id":192,"known_for_department":"Acting","name":"Morgan Freeman","original_name":"Morga

In [234]:
type(credits[0])

bs4.BeautifulSoup

In [277]:
# List for saving cleaned data
data = []

for credits_soup in credits:
    # Convert the Beautiful Soup object to a dictionary
    credit = json.loads(str(credits_soup))

    # Extract actors names
    actors = [cast['name'] for cast in credit.get('cast', [])[:3]]

    # Extract the director's name
    director = next((crew['name'] for crew in credit.get('crew', []) if crew.get('job') == 'Director'), None)

    # Append data to the list
    data.append({'id': credit['id'], 'actor_1': actors[0] if len(actors) > 0 else None, 
                 'actor_2': actors[1] if len(actors) > 1 else None, 
                 'actor_3': actors[2] if len(actors) > 2 else None,
                 'director': director})

# Create a DataFrame
movie_credits = pd.DataFrame(data)

# Display the DataFrame
movie_credits.head()

,id,actor_1,actor_2,actor_3,director
0,384521,Gugu Mbatha-Raw,Daniel Brühl,Chris O'Dowd,Julius Onah
1,9665,Matthew Broderick,Denzel Washington,Cary Elwes,Edward Zwick
2,22881,Sandra Bullock,Tim McGraw,Quinton Aaron,John Lee Hancock
3,14748,Andrew Garfield,Katie Lyons,Peter Mullan,John Crowley
4,2059,Nicolas Cage,Diane Kruger,Justin Bartha,Jon Turteltaub


In [253]:
print(movie_credits.head(10))

       id                                             actors  \
0  384521      [Gugu Mbatha-Raw, Daniel Brühl, Chris O'Dowd]   
1    9665  [Matthew Broderick, Denzel Washington, Cary El...   
2   22881        [Sandra Bullock, Tim McGraw, Quinton Aaron]   
3   14748       [Andrew Garfield, Katie Lyons, Peter Mullan]   
4    2059        [Nicolas Cage, Diane Kruger, Justin Bartha]   
5  453755  [Mads Mikkelsen, Maria Thelma Smáradóttir, Tin...   
6  677179  [Michael B. Jordan, Tessa Thompson, Jonathan M...   
7   14359  [Meryl Streep, Philip Seymour Hoffman, Amy Adams]   
8    7985  [Christina Ricci, James McAvoy, Catherine O'Hara]   
9   11302     [Woody Allen, Louise Lasser, Carlos Montalbán]   

               director  
0           Julius Onah  
1          Edward Zwick  
2      John Lee Hancock  
3          John Crowley  
4        Jon Turteltaub  
5             Joe Penna  
6     Michael B. Jordan  
7  John Patrick Shanley  
8         Mark Palansky  
9           Woody Allen  


In [ ]:
from sqlalchemy import create_engine, Column, Integer, String, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, Session
import pandas as pd

# Create a SQLite database engine
engine = create_engine('sqlite:///movie_database.db', echo=True)

# Create a base class for declarative models
Base = declarative_base()

class Movie(Base):
    __tablename__ = 'movies'
    id = Column(Integer, primary_key=True)
    title = Column(String)
    
    credits = relationship('Credit', back_populates='movie')

# Define the Credits model
class Credit(Base):
    __tablename__ = 'credits'

    id = Column(Integer, primary_key=True)
    movie_id = Column(Integer, ForeignKey('movies.id'))
    actors = Column(Text)
    director = Column(String)

# Establish the foreign key relationship
Movie.credits = relationship('Credit', back_populates='movie')
Credit.movie = relationship('Movie', back_populates='credits')

# Create tables in the database
Base.metadata.create_all(engine)

# Assuming you already have dataframes df_movies and df_credits
# Load data into the tables

movie_credits.to_sql('credits', engine, if_exists='replace', index=False)

In [268]:
type(keywords[0])

bs4.BeautifulSoup

In [270]:
data = []

for soup in keywords:
    # Convert the Beautiful Soup object to a dictionary
    keywords_data = json.loads(str(soup))
    keyword = ", ".join(key["name"] for key in keywords_data["keywords"])
    data.append({"id": keywords_data["id"], "keywords": keyword})

# Creating a DataFrame
movie_keywords = pd.DataFrame(data)

# Display the DataFrame
print(movie_keywords)

          id                                           keywords
0     384521  experiment, disorientation, alien, space, scie...
1       9665  racism, battle, union soldier, confederate sol...
2      22881  sports, american football, adoption, private s...
3      14748  child abuse, rape, based on novel or book, par...
4       2059  new york city, treasure, philadelphia, pennsyl...
...      ...                                                ...
4995    3050  based on novel or book, tiger, san francisco, ...
4996    2899  egypt, magic, palace, civil engineer, cleopatr...
4997   11619   ship, london, england, return, girlfriend, rubin
4998  282984                              college, rhode island
4999    2064  chicago, illinois, coma, sibling relationship,...

[5000 rows x 2 columns]


In [278]:
from sqlalchemy import create_engine, Column, Integer, String, Float, DateTime, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker
import pandas as pd

# Create engine and connect to the database
engine = create_engine('sqlite:///movie_database.db', echo=True)
Base = declarative_base()

# Define MovieMetadata table
class MovieMetadata(Base):
    __tablename__ = 'metadata'

    id = Column(Integer, primary_key=True)
    imdb_id = Column(String, unique=True)
    title = Column(String)
    release_date = Column(DateTime)
    vote_average = Column(Float)
    genres = Column(String)
    overview = Column(String)
    tagline = Column(String)
    popularity = Column(Float)
    revenue = Column(Integer)
    runtime = Column(Integer)
    spoken_languages = Column(String)
    vote_count = Column(Integer)

# Define MovieCredits table
class MovieCredits(Base):
    __tablename__ = 'credits'

    id = Column(Integer, primary_key=True)
    actor_1 = Column(String)
    actor_2 = Column(String)
    actor_3 = Column(String)
    director = Column(String)
    movie_id = Column(Integer, ForeignKey('metadata.id'))

# Define MovieKeywords table
class MovieKeywords(Base):
    __tablename__ = 'keywords'

    id = Column(Integer, primary_key=True)
    keywords = Column(String)
    movie_id = Column(Integer, ForeignKey('metadata.id'))

# Bind the engine to the Base
Base.metadata.create_all(engine)

# Create a session to interact with the database
Session = sessionmaker(bind=engine)
session = Session()

# Insert data into MovieMetadata table
movie_metadata.to_sql('metadata', engine, if_exists='replace', index=False)

# Insert data into MovieCredits table
movie_credits.to_sql('credits', engine, if_exists='replace', index=False)

# Insert data into MovieKeywords table
movie_keywords.to_sql('keywords', engine, if_exists='replace', index=False)

# Commit the changes
session.commit()

# Close the session
session.close()


2024-01-18 18:21:59,305 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2024-01-18 18:21:59,309 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("metadata")
2024-01-18 18:21:59,311 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-18 18:21:59,314 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("credits")
2024-01-18 18:21:59,315 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-18 18:21:59,318 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("keywords")
2024-01-18 18:21:59,322 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-18 18:21:59,326 INFO sqlalchemy.engine.Engine COMMIT
2024-01-18 18:21:59,347 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("metadata")
2024-01-18 18:21:59,348 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-18 18:21:59,356 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("metadata")
2024-01-18 18:21:59,359 INFO sqlalchemy.engine.Engine [raw sql] ()
2024-01-18 18:21:59,364 INFO sqlalchemy.engine.Engine SELECT name FROM sqlite_master WHER

2024-01-18 18:21:59,777 INFO sqlalchemy.engine.Engine INSERT INTO metadata (id, imdb_id, title, release_date, vote_average, genres, overview, tagline, popularity, revenue, runtime, spoken_languages, vote_count) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
2024-01-18 18:21:59,782 INFO sqlalchemy.engine.Engine [generated in 0.23287s] ((384521, 'tt2548396', 'The Cloverfield Paradox', '2018-02-04', 5.598, 'Horror, Science Fiction, Action, Thriller', 'Orbiting above a planet on the brink of war, scientists test a device to solve an energy crisis and end up face-to-face with a dark alternate reality.', 'The future unleashed every thing', 18.481, 0, 102, 'English', 3057), (9665, 'tt0097441', 'Glory', '1989-12-15', 7.485, 'Drama, History, War', "Robert Gould Shaw leads the US Civil War's first all-black volunteer company, fighting prejudices of both his own Union army and the Confederates.", 'Their innocence. Their heritage. Their lives.', 20.379, 26800000, 122, 'English', 1475), (22881, 'tt